# QMM Unified Workflow

This is the single clean notebook for the `QMM` folder. It uses only files inside `QMM`: the local `src/qmm` package, the local example JSON inputs, and the curated `results/` tables copied into this directory.

The notebook is organized in the same order as the physics workflow:

1. symmetric nuclear matter at saturation
2. symmetric finite-temperature critical point
3. symmetric quarkyonic matter
4. asymmetric hadronic calibration
5. asymmetric quarkyonic matter at fixed `y` and in beta equilibrium
6. neutron-star structure from the beta-equilibrium EOS
7. how to run your own real-gas model


In [ ]:
from pathlib import Path
import sys
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Markdown


def locate_root() -> Path:
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (candidate / 'src' / 'qmm').exists():
            return candidate.resolve()
    raise RuntimeError('Could not locate the QMM project root from this notebook.')


ROOT = locate_root()
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from qmm.models import MODELS
from qmm.sound_speed import compute_vs2_only

RESULTS = ROOT / 'results'
EXAMPLES = ROOT / 'examples'

plt.style.use('default')
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print(f'ROOT = {ROOT}')
print('Available built-in models:', ', '.join(sorted(MODELS)))


## Physics Map

The workflow enforced in `QMM` is:

### 1. Symmetric nuclear matter at saturation

For a chosen real-gas model, the code solves for the couplings `a` and `b` using the ground-state conditions

`P(n0) = 0`, `E/A(n0) - m_N = -16 MeV`.

After that, the incompressibility is evaluated as

`K0 = 9 (dP/dn_B)|_(n0)`.

### 2. Finite-temperature critical point

The critical point is found from the symmetric finite-temperature EOS by solving

`(dP/dn_B)_T = 0`, `(d^2 P/dn_B^2)_T = 0`.

This gives `Tc`, `nc`, and `Pc`.

### 3. Symmetric quarkyonic matter

At fixed baryon density `n_B`, the code minimizes the zero-temperature energy density `epsilon(n_B, f_Q)` with respect to the quark fraction `f_Q`. The shell parameter is set by `lambda_momentum_mev`.

The sound speed is reconstructed from the energy-density curve using

`mu_B = d epsilon / d n_B`, `P = n_B mu_B - epsilon`, `v_s^2 = (dP/dn_B) / mu_B`.

### 4. Asymmetric matter

The asymmetric hadronic extension is fitted from the five constraints used in this project:

- binding energy `E/A = -16 MeV`
- saturation density `n0 = 0.16 fm^-3`
- symmetry energy `J = 32.5 MeV`
- symmetry-energy slope `L = 58.9 MeV`
- incompressibility target or scan range `K0`

The two branches already implemented are:

- `equal_b`: `b_pn = b_n`
- `target_l`: `b_pn != b_n`, adjusted to reproduce `L`

### 5. Beta equilibrium and neutron stars

For the quarkyonic asymmetric EOS, the notebook uses the same local code that solves charge neutrality with electrons and muons, then integrates the TOV equations with an RK4 solver using the monotonic beta-equilibrium EOS.


In [ ]:
def load_csv(relative_path: str) -> pd.DataFrame:
    return pd.read_csv(ROOT / relative_path)


def show_config(name: str) -> None:
    with (EXAMPLES / name).open('r', encoding='utf-8') as handle:
        print(f'===== {name} =====')
        print(json.dumps(json.load(handle), indent=2))
        print()


def model_label(model: str, parameter_value: float | None) -> str:
    if parameter_value is None or (isinstance(parameter_value, float) and np.isnan(parameter_value)):
        return model
    return f'{model} ({parameter_value:.3g})'


def load_fixed_y_bundle(run_dir: str, prefix: str):
    base = ROOT / run_dir
    fixed = {}
    for path in sorted(base.glob(f'{prefix}_asymmetric_y_*.csv')):
        y_value = float(path.stem.split('_y_')[-1])
        fixed[y_value] = pd.read_csv(path)
    beta_path = base / f'{prefix}_asymmetric_beta.csv'
    beta = pd.read_csv(beta_path) if beta_path.exists() else None
    return fixed, beta


## Example Inputs

These JSON files are the clean entry points for rerunning the main workflows. They write new outputs into `results/generated/` and do not overwrite the curated reference tables used below.


In [ ]:
for example_name in sorted(path.name for path in EXAMPLES.glob('*.json')):
    show_config(example_name)


## 1. Symmetric Nuclear Matter At Saturation

The table below contains the solved ground-state parameters for the surveyed models. For models with an extra parameter, the value used in the fit is kept in `parameter_value`.


In [ ]:
ground = load_csv('results/nuclear_matter/combined_ground_state_results.csv')
ground['parameter_value'] = pd.to_numeric(ground['parameter_value'], errors='coerce')
ground['label'] = [model_label(row.model, row.parameter_value) for row in ground.itertuples()]

display(
    ground[
        ['label', 'a', 'b', 'K0', 'binding', 'n_sat', 'p_sat']
    ].sort_values('K0', ascending=False).reset_index(drop=True)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

scatter = axes[0].scatter(ground['b'], ground['a'], c=ground['K0'], cmap='viridis', s=90)
for row in ground.itertuples():
    axes[0].annotate(row.label, (row.b, row.a), xytext=(4, 4), textcoords='offset points', fontsize=8)
axes[0].set_xlabel(r'$b$ [fm$^3$]')
axes[0].set_ylabel(r'$a$ [MeV fm$^3$]')
axes[0].set_title('Ground-state couplings')
fig.colorbar(scatter, ax=axes[0], label=r'$K_0$ [MeV]')

axes[1].bar(ground['label'], ground['K0'], color='#1f77b4')
axes[1].axhspan(250.0, 315.0, color='#ffcc99', alpha=0.35, label=r'target $K_0$ band')
axes[1].set_ylabel(r'$K_0$ [MeV]')
axes[1].set_title('Incompressibility by model')
axes[1].tick_params(axis='x', rotation=60)
axes[1].legend(loc='upper right')

plt.show()


## 2. Symmetric Finite-Temperature Critical Point

The next table and plots summarize the `T_c`, `n_c`, and `P_c` solutions obtained from the finite-temperature SCF solver.


In [ ]:
quantum = load_csv('results/nuclear_matter/combined_quantum_results.csv')
quantum['parameter_value'] = pd.to_numeric(quantum['parameter_value'], errors='coerce')
quantum['label'] = [model_label(row.model, row.parameter_value) for row in quantum.itertuples()]

display(
    quantum[
        ['label', 'K0', 'Tc', 'nc', 'Pc', 'iterations', 'score']
    ].sort_values('Tc', ascending=False).reset_index(drop=True)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

sc0 = axes[0].scatter(quantum['nc'], quantum['Tc'], c=quantum['K0'], cmap='plasma', s=110)
for row in quantum.itertuples():
    axes[0].annotate(row.label, (row.nc, row.Tc), xytext=(4, 4), textcoords='offset points', fontsize=8)
axes[0].set_xlabel(r'$n_c$ [fm$^{-3}$]')
axes[0].set_ylabel(r'$T_c$ [MeV]')
axes[0].set_title(r'$T_c$ vs $n_c$')
fig.colorbar(sc0, ax=axes[0], label=r'$K_0$ [MeV]')

sc1 = axes[1].scatter(quantum['nc'], quantum['K0'], c=quantum['Tc'], cmap='viridis', s=110)
for row in quantum.itertuples():
    axes[1].annotate(row.label, (row.nc, row.K0), xytext=(4, 4), textcoords='offset points', fontsize=8)
axes[1].set_xlabel(r'$n_c$ [fm$^{-3}$]')
axes[1].set_ylabel(r'$K_0$ [MeV]')
axes[1].set_title(r'$K_0$ vs $n_c$')
fig.colorbar(sc1, ax=axes[1], label=r'$T_c$ [MeV]')

plt.show()


## 3. Symmetric Quarkyonic Matter

The combined survey below uses the local `QMM/results/quarkyonic/symmetric/` files. In the survey curves, the standard quarkyonic default is `\lambda=300 MeV`. For the dedicated CS and TVM runs, the local examples keep `\lambda=200 MeV`, matching the calibration used in the original CS/TVM asymmetric workflow.


In [ ]:
q_summary = load_csv('results/quarkyonic/symmetric/combined_quarkyonic_sound_speed_summary.csv')
q_curves = load_csv('results/quarkyonic/symmetric/combined_quarkyonic_sound_speed_curves.csv')

display(q_summary.sort_values('n_tr_over_n0').reset_index(drop=True))

survey_models = ['vdw', 'rks', 'pr', 'dieterici', 'clausius']
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)

for model in survey_models:
    df = q_curves[q_curves['model'] == model]
    axes[0].plot(df['n_over_n0'], df['vs2'], lw=2, label=model)
    axes[1].plot(df['n_over_n0'], df['quark_fraction'], lw=2, label=model)

axes[0].axhline(1.0 / 3.0, color='gray', ls=(0, (5, 5)), lw=1.2)
axes[0].set_ylim(-0.1, 0.8)
axes[0].set_xlabel(r'$n/n_0$')
axes[0].set_ylabel(r'$v_s^2$')
axes[0].set_title('Symmetric quarkyonic sound speed')
axes[0].legend(loc='upper right')

axes[1].set_xlabel(r'$n/n_0$')
axes[1].set_ylabel(r'$f_Q$')
axes[1].set_title('Quark fraction')
axes[1].legend(loc='upper left')

plt.show()


In [ ]:
symmetric_runs = {
    'CS, $\lambda=200$ MeV': load_csv('results/asymmetric/cs_full/cs_full_symmetric_quarkyonic_curve.csv'),
    'TVM, $\lambda=200$ MeV': load_csv('results/asymmetric/tvm_full/tvm_full_symmetric_quarkyonic_curve.csv'),
    'Clausius, $\lambda=300$ MeV': load_csv('results/asymmetric/clausius_reference/clausius_full_symmetric_quarkyonic_curve.csv'),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)
for label, df in symmetric_runs.items():
    axes[0].plot(df['n_over_n0'], df['vs2'], lw=2, label=label)
    axes[1].plot(df['n_over_n0'], df['quark_fraction'], lw=2, label=label)

axes[0].axhline(1.0 / 3.0, color='gray', ls=(0, (5, 5)), lw=1.2)
axes[0].set_ylim(-0.1, 0.8)
axes[0].set_xlabel(r'$n/n_0$')
axes[0].set_ylabel(r'$v_s^2$')
axes[0].set_title('Dedicated symmetric reference curves')
axes[0].legend(loc='upper right')

axes[1].set_xlabel(r'$n/n_0$')
axes[1].set_ylabel(r'$f_Q$')
axes[1].set_title('Dedicated quark fractions')
axes[1].legend(loc='upper left')

plt.show()


### Direct `numpy.gradient` Sound-Speed Check

The workflow itself stores the smoothed reconstruction of `v_s^2`. The next cell uses the simple diagnostic function

```python
mu_b = np.gradient(E, n, edge_order=2)
mu_bb = np.gradient(mu_b, n, edge_order=2)
vs2 = n * mu_bb / mu_b
```

on the local CS and TVM symmetric curves, so you can compare the direct derivative estimate with the stored workflow result.


In [ ]:
diagnostic_runs = {
    'CS': load_csv('results/asymmetric/cs_full/cs_full_symmetric_quarkyonic_curve.csv'),
    'TVM': load_csv('results/asymmetric/tvm_full/tvm_full_symmetric_quarkyonic_curve.csv'),
}

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), constrained_layout=True)
for ax, (label, df) in zip(axes, diagnostic_runs.items()):
    vs2_direct = compute_vs2_only(df['n'].to_numpy(), df['eps'].to_numpy())
    ax.plot(df['n_over_n0'], df['vs2'], lw=2.2, label='workflow reconstruction')
    ax.plot(df['n_over_n0'], vs2_direct, lw=1.6, ls='--', label='direct numpy.gradient')
    ax.axhline(1.0 / 3.0, color='gray', ls=(0, (5, 5)), lw=1.0)
    ax.set_ylim(-0.1, 0.8)
    ax.set_xlabel(r'$n/n_0$')
    ax.set_ylabel(r'$v_s^2$')
    ax.set_title(label)
    ax.legend(loc='upper right')

plt.show()


## 4. Asymmetric Hadronic Calibration

        The table below summarizes the asymmetric couplings already solved in the local workflow. It includes both branches:

        - `equal_b`: `b_{pn}=b_n`
        - `target_l`: `b_{pn}
eq b_n`, tuned to reproduce `L=58.9 MeV`


In [ ]:
asym = load_csv('results/nuclear_matter/asymmetric_branch_summary.csv')
asym['tag'] = asym['model'] + ' / ' + asym['branch_mode']

display(
    asym[
        ['tag', 'a_n', 'a_pn', 'b_n', 'b_pn', 'ratio_a_pn_over_a_n', 'ratio_b_pn_over_b_n', 'J', 'L', 'K0']
    ].reset_index(drop=True)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)

axes[0].bar(asym['tag'], asym['ratio_a_pn_over_a_n'], color='#2a9d8f')
axes[0].set_ylabel(r'$a_{pn}/a_n$')
axes[0].set_title('Attractive-coupling split')
axes[0].tick_params(axis='x', rotation=55)

axes[1].bar(asym['tag'], asym['ratio_b_pn_over_b_n'], color='#e76f51')
axes[1].set_ylabel(r'$b_{pn}/b_n$')
axes[1].set_title('Excluded-volume split')
axes[1].tick_params(axis='x', rotation=55)

plt.show()


## 5. Asymmetric Quarkyonic Matter At Fixed `y`

The next plots collect the fixed-proton-fraction runs for CS and TVM. These are the local curves already generated with `\lambda=200 MeV` and the `target_l` asymmetric branch.


In [ ]:
fixed_runs = {
    'CS': ('results/asymmetric/cs_full', 'cs_full'),
    'TVM': ('results/asymmetric/tvm_full', 'tvm_full'),
}

fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True, sharex='col')

for row_index, (label, payload) in enumerate(fixed_runs.items()):
    run_dir, prefix = payload
    fixed_profiles, _ = load_fixed_y_bundle(run_dir, prefix)
    ax_fq = axes[row_index, 0]
    ax_vs = axes[row_index, 1]

    for y_value, df in fixed_profiles.items():
        curve_label = f'y={y_value:.1f}'
        ax_fq.plot(df['n_over_n0'], df['quark_fraction'], lw=1.8, label=curve_label)
        ax_vs.plot(df['n_over_n0'], df['vs2'], lw=1.8, label=curve_label)

    ax_fq.set_ylabel(f'{label}: $f_Q$')
    ax_fq.set_title(f'{label} fixed-y quark fraction')

    ax_vs.set_ylabel(f'{label}: $v_s^2$')
    ax_vs.set_ylim(-0.1, 0.8)
    ax_vs.axhline(1.0 / 3.0, color='gray', ls=(0, (5, 5)), lw=1.0)
    ax_vs.set_title(f'{label} fixed-y sound speed')

axes[1, 0].set_xlabel(r'$n/n_0$')
axes[1, 1].set_xlabel(r'$n/n_0$')
axes[0, 1].legend(loc='upper right', ncol=2)

plt.show()


## 6. Beta Equilibrium With Leptons

In the local implementation, the beta-equilibrium branch solves for the proton fraction `y` by imposing charge neutrality with electrons and muons. Once the neutral configuration is found at each density, the code recomputes the EOS, sound speed, and quark fraction.


In [ ]:
beta_runs = {
    'CS': load_csv('results/asymmetric/cs_full/cs_full_asymmetric_beta.csv'),
    'TVM': load_csv('results/asymmetric/tvm_full/tvm_full_asymmetric_beta.csv'),
    'Clausius': load_csv('results/asymmetric/clausius_reference/clausius_full_asymmetric_beta.csv'),
}

fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True, sharex=True)

for label, df in beta_runs.items():
    axes[0, 0].plot(df['n_over_n0'], df['y'], lw=2, label=label)
    axes[0, 1].plot(df['n_over_n0'], df['quark_fraction'], lw=2, label=label)
    axes[1, 0].plot(df['n_over_n0'], df['vs2'], lw=2, label=label)
    axes[1, 1].plot(df['n_over_n0'], df['mu_e'], lw=2, label=label)

axes[0, 0].set_ylabel('proton fraction y')
axes[0, 0].set_title('Beta-equilibrium proton fraction')

axes[0, 1].set_ylabel(r'$f_Q$')
axes[0, 1].set_title('Beta-equilibrium quark fraction')

axes[1, 0].set_ylabel(r'$v_s^2$')
axes[1, 0].set_title('Beta-equilibrium sound speed')
axes[1, 0].set_ylim(-0.1, 0.8)
axes[1, 0].axhline(1.0 / 3.0, color='gray', ls=(0, (5, 5)), lw=1.0)

axes[1, 1].set_ylabel(r'$\mu_e$ [MeV]')
axes[1, 1].set_title('Leptonic chemical potential')

axes[1, 0].set_xlabel(r'$n/n_0$')
axes[1, 1].set_xlabel(r'$n/n_0$')
axes[0, 1].legend(loc='upper left')

plt.show()


## 7. Neutron Stars From The Beta-Equilibrium EOS

The local neutron-star solver first extracts a monotonic beta-equilibrium EOS, converts pressure and energy density to CGS units, interpolates the EOS in log-space, and integrates the TOV equations with an RK4 stepper.


In [ ]:
ns_summary = load_csv('results/neutron_stars/cs_tvm_beta_neutron_star_summary.csv')
display(ns_summary)

ns_runs = {
    'CS': load_csv('results/asymmetric/cs_full/cs_full_neutron_star_sequence.csv'),
    'TVM': load_csv('results/asymmetric/tvm_full/tvm_full_neutron_star_sequence.csv'),
}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6), constrained_layout=True)

for label, df in ns_runs.items():
    axes[0].plot(df['radius_km'], df['mass_msun'], lw=2.2, label=label)
    axes[1].plot(df['central_energy_density_mev_fm3'], df['mass_msun'], lw=2.2, label=label)

axes[0].set_xlabel('radius [km]')
axes[0].set_ylabel(r'mass [$M_\odot$]')
axes[0].set_title('Mass-radius relation')
axes[0].legend(loc='lower left')

axes[1].set_xlabel(r'central energy density [MeV fm$^{-3}$]')
axes[1].set_ylabel(r'mass [$M_\odot$]')
axes[1].set_title('Mass vs central energy density')
axes[1].legend(loc='lower right')

plt.show()


## 8. How To Run Your Own Real-Gas Model

There are two levels:

### A. Run one of the built-in models

From the `QMM` directory:

```bash
python3 -m pip install -e .
python3 -m qmm examples/cs_full.json
python3 -m qmm examples/tvm_full.json
python3 -m qmm examples/clausius_reference.json
```

### B. Add a new model

Edit `src/qmm/models.py` and add one more `InteractionModel`. The required ingredients are:

1. `U(n,b,parameter)`
2. `dU(n,b,parameter)`
3. `nid_from_n(n,b)`
4. `n_from_nid(n_id,b)`
5. `volume_fraction(n,b)`
6. `species_volume_fraction(x)`
7. `pressure_prefactor(n,b)`

A minimal template looks like this:

```python
def u_my_model(n, b, parameter=None):
    ...

def du_my_model(n, b, parameter=None):
    ...

def volume_fraction_my_model(n, b):
    ...

def species_fraction_my_model(x_value):
    ...

def nid_from_n_my_model(n, b):
    ...

def n_from_nid_my_model(n_id, b):
    ...

def pressure_prefactor_my_model(n, b):
    ...

MODELS['my_model'] = InteractionModel(
    name='my_model',
    parameter_name='my_parameter',
    parameter_range=(min_value, max_value),
    supports_quantum=True,
    supports_symmetric_quarkyonic=True,
    supports_asymmetric_hadronic=True,
    supports_asymmetric_quarkyonic=True,
    U=u_my_model,
    dU=du_my_model,
    nid_from_n=nid_from_n_my_model,
    n_from_nid=n_from_nid_my_model,
    volume_fraction=volume_fraction_my_model,
    species_volume_fraction=species_fraction_my_model,
    pressure_prefactor=pressure_prefactor_my_model,
)
```

Then create a JSON input in `examples/` with your chosen workflow blocks. If your model has an extra free parameter, supply either a fixed `parameter_value` or a `parameter_search` block with the target `K0`.


In [ ]:
print('Built-in models available in this clean package:')
for name in sorted(MODELS):
    print(' -', name)
